# 🧑‍🍳 Neosantara AI Cookbook: Jev Decision Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/jev-decision-models.ipynb)

Call TypeSafe **Jev**, a System One decision model, through Neosantara with your `NEOSANTARA_API_KEY`. Jev returns typed decisions with probabilities instead of text:

| Type | Answer | Example |
| :--- | :--- | :--- |
| `noul` | P(True) | Does this ticket need a human agent? |
| `score` | Probability per ordered level | How severe is the impact? |
| `choice` | Probability per option | Which team handles it? |

Jev is not available on the Free tier. Input costs $0.042 per 1M tokens and output tokens are free.

### 📖 Documentation
- [Jev-style decision programs with DSPy](https://docs.neosantara.xyz/en/guides/dspy-decision-programs)
- [Model catalog](https://docs.neosantara.xyz/en/gateway/models)

In [ ]:
!pip install -q requests

import os
from getpass import getpass

try:
    from google.colab import userdata
    NEOSANTARA_API_KEY = userdata.get("NEOSANTARA_API_KEY")
    if not NEOSANTARA_API_KEY:
        raise ValueError("Secret not found")
    os.environ["NEOSANTARA_API_KEY"] = NEOSANTARA_API_KEY
    print("✅ API Key loaded from Google Colab Secrets!")
except (ImportError, Exception):
    if "NEOSANTARA_API_KEY" not in os.environ:
        print("🔑 API Key not found in Secrets. Please paste it below:")
        os.environ["NEOSANTARA_API_KEY"] = getpass("Neosantara API Key: ")
    else:
        print("✅ API Key loaded from environment!")

## 1. Ask Jev three questions in one request

`POST /v1/systemone` uses the TypeSafe System One wire format. `state` is the data Jev judges; each entry in `questions` is one decision.

In [ ]:
import json
import requests

BASE_URL = "https://api.neosantara.xyz"
HEADERS = {
    "Authorization": f"Bearer {os.environ['NEOSANTARA_API_KEY']}",
    "Content-Type": "application/json",
}

QUESTIONS = {
    "escalate": {
        "type": "noul",
        "instructions": "Apakah tiket ini perlu ditangani agen manusia?",
        "criteria": {"true": "Butuh agen manusia", "false": "Bisa dibalas otomatis"},
    },
    "severity": {
        "type": "score",
        "instructions": "Seberapa besar dampaknya ke pelanggan?",
        "criteria": ["Pertanyaan umum", "Transaksi terganggu", "Uang pelanggan tertahan"],
    },
    "team": {
        "type": "choice",
        "instructions": "Tim mana yang harus menangani tiket ini?",
        "criteria": {
            "pembayaran": "QRIS, virtual account, e-wallet, refund",
            "pengiriman": "Resi, kurir, paket telat atau hilang",
            "akun": "Login, OTP, verifikasi KYC",
            "teknis": "Aplikasi error, crash, fitur tidak jalan",
        },
    },
}


def ask_jev(ticket: str) -> dict:
    res = requests.post(
        f"{BASE_URL}/v1/systemone",
        headers=HEADERS,
        json={"model": "jev", "state": ticket, "questions": QUESTIONS},
        timeout=30,
    )
    res.raise_for_status()
    return res.json()


result = ask_jev("Min, saldo udah kepotong pas bayar QRIS tapi status order masih menunggu pembayaran.")
print(json.dumps(result, indent=2, ensure_ascii=False))

## 2. Turn probabilities into actions

Jev only returns probabilities. Your code owns the thresholds, so routing stays deterministic and auditable. A `threshold` below 0.5 escalates more readily; a `weight` above 1.0 favors an option when the model is unsure.

In [ ]:
ESCALATE_THRESHOLD = 0.3
SEVERITY_CUTS = [0.5, 1.5]
TEAM_WEIGHTS = {"pembayaran": 1.2}


def decide(answers: dict) -> dict:
    p_escalate = answers["escalate"]["noul"]
    severity_level = sum(1 for cut in SEVERITY_CUTS if answers["severity"]["score"] >= cut)
    team_probs = answers["team"]["probabilities"]
    team = max(team_probs, key=lambda k: team_probs[k] * TEAM_WEIGHTS.get(k, 1.0))
    return {
        "action": "antrian_agen" if p_escalate >= ESCALATE_THRESHOLD else "balas_otomatis",
        "p_escalate": p_escalate,
        "severity": severity_level,
        "team": team,
    }


tickets = [
    "Min, saldo udah kepotong pas bayar QRIS tapi status order masih menunggu pembayaran.",
    "Kak, resi JNE saya kok belum update dari kemarin?",
    "Kode OTP ga masuk-masuk, udah coba 3 kali.",
]
for ticket in tickets:
    print(ticket)
    print("  ->", decide(ask_jev(ticket)["answers"]))

## 3. Vercel AI Gateway format

`POST /v1/evaluate` accepts the same questions in the Vercel AI Gateway format. Boolean questions use `"type": "boolean"` and return `probability`.

In [ ]:
res = requests.post(
    f"{BASE_URL}/v1/evaluate",
    headers=HEADERS,
    json={
        "model": "jev",
        "state": "Kode OTP ga masuk-masuk, udah coba 3 kali.",
        "questions": {
            "escalate": {"type": "boolean", "instructions": "Apakah tiket ini perlu ditangani agen manusia?"}
        },
    },
    timeout=30,
)
print(res.status_code, json.dumps(res.json(), indent=2, ensure_ascii=False))

## Next steps

| Task | Recipe |
| :--- | :--- |
| Declare the same decisions as DSPy signatures and calibrate thresholds with `ReAnchor` | [DSPy Decision Types](https://colab.research.google.com/github/neosantara-xyz/examples/blob/main/cookbook/advanced/dspy-decision-types.ipynb) |
| Read the full guide | [Jev-style decision programs](https://docs.neosantara.xyz/en/guides/dspy-decision-programs) |